- Activate environment
```
conda create -n experiment python=3.12
conda activate experiment
pip install -r requirements.txt
```

- Download test files
```
curl https://iotanalytics.unsw.edu.au/iottestbed/pcap/filelist.txt -o filelist.txt
head -20 filelist.txt | grep -Ev '(^#.*|^$)' filelist.txt | xargs -n 1 curl -O
```

- Untar file
```
for f in *.zip; do
    unzip "$f"
done
```

In [1]:
import pandas as pd
import os
from typing import Tuple, Set, Dict
from pathlib import Path

- Load the data

In [2]:
CSV_DATASET_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv"
CSV_PROCESSED_DATA_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/processed-data/from-csv"
PCAPC_DATASET_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap"
PCAP_PROCESSED_DATA_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/processed-data/from-pcap"
DEVICE_MAPPING_FILE = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/device-mapping/List_Of_Devices.txt"

- Assuming the last 9 devices are non-IoT with some caveats
    - These 9 devices include **Nest DropCam** which is not considered in the dataset because of its special traffic
    - **TP Link Router** is also will be taken off from the list.
- The final list will have 21 IoT devices and 7 non-IoT devices
    - The mapping will print 22 IoT devices because the Insteon Camera has 2 mapping for Wired and Wireless connection

In [3]:
def process_device_mapping(file_path: str) -> Tuple[Set[str], Set[str]]:
    """
    Process the device mapping file and return a set of IoT devices and a set of non-IoT devices.
    :param file_path: The path to the device mapping file.
    :return: A tuple containing a set of IoT devices and a set of non-IoT devices.
    """
    device_mapping = []
    with open(file_path, 'r') as f:
        for line in f:
            line_data = line.strip().split('\t')
            mapping = [data.strip() for data in line_data if data.strip()]
            if mapping and mapping[0].lower() != "list of devices":
                device_mapping.append((mapping[0], mapping[1]))

    deviceToBeRemoved = set(["14:cc:20:51:33:ea", "30:8c:fb:b6:ea:45"])
    all_devices = {mapping[1]: mapping[0] for mapping in device_mapping if mapping[1] not in deviceToBeRemoved}
    iot_devices = [mapping[0] for mapping in device_mapping[:-9] if mapping[1] not in deviceToBeRemoved]
    non_iot_devices = [mapping[0] for mapping in device_mapping[-9:] if mapping[1] not in deviceToBeRemoved]

    print(f"{len(iot_devices)} IoT devices: {iot_devices}")
    print(f"{len(non_iot_devices)} Non-IoT devices: {non_iot_devices}")
    print(f"{len(all_devices)} total devices: {all_devices}")

    return all_devices, iot_devices, non_iot_devices

all_devices, iot_devices, non_iot_devices = process_device_mapping(DEVICE_MAPPING_FILE)

22 IoT devices: ['Smart Things', 'Amazon Echo', 'Netatmo Welcome', 'TP-Link Day Night Cloud camera', 'Samsung SmartCam', 'Dropcam', 'Insteon Camera', 'Insteon Camera', 'Withings Smart Baby Monitor', 'Belkin Wemo switch', 'TP-Link Smart plug', 'iHome', 'Belkin wemo motion sensor', 'NEST Protect smoke alarm', 'Netatmo weather station', 'Withings Smart scale', 'Blipcare Blood Pressure meter', 'Withings Aura smart sleep sensor', 'Light Bulbs LiFX Smart Bulb', 'Triby Speaker', 'PIX-STAR Photo-frame', 'HP Printer']
7 Non-IoT devices: ['Samsung Galaxy Tab', 'Android Phone', 'Laptop', 'MacBook', 'Android Phone', 'IPhone', 'MacBook/Iphone']
29 total devices: {'d0:52:a8:00:67:5e': 'Smart Things', '44:65:0d:56:cc:d3': 'Amazon Echo', '70:ee:50:18:34:43': 'Netatmo Welcome', 'f4:f2:6d:93:51:f1': 'TP-Link Day Night Cloud camera', '00:16:6c:ab:6b:88': 'Samsung SmartCam', '30:8c:fb:2f:e4:b2': 'Dropcam', '00:62:6e:51:27:2e': 'Insteon Camera', 'e8:ab:fa:19:de:4f': 'Insteon Camera', '00:24:e4:11:18:a8': '

In [4]:
files = [f"{CSV_DATASET_FOLDER}/{f.name}" for f in sorted(Path(CSV_DATASET_FOLDER).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in files:
    print(file)

../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-23.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-24.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-25.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-26.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-27.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-28.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-29.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-30.csv
../../dataset/identify-iot-devices-and-events-based-on-p

- The `TIME` columns in already in Unix time seconds format
- According to the paper, we will be grouping traffic with the same second window and originating from the same MAC source address. Based on the IoT and non-IoT devices list above, we'll label this traffic if it is IoT or non-IoT

In [ ]:
def preprocess_csv(file_path: str, all_devices: Dict[str, str]) -> pd.DataFrame:
    """
    Preprocess the CSV file and return a DataFrame with the relevant features.
    :param file_path: The path to the CSV file.
    :param all_devices: A dictionary mapping device MAC addresses to names.
    :return: A preprocessed DataFrame with the relevant features.
    """
    df = pd.read_csv(file_path)
    
    # First map eth.src to device name
    df["device_name"] = df["eth.src"].map(all_devices).fillna("Unknown")
    filtered_unknown_df = df[df["device_name"] != "Unknown"]
    features_df = (filtered_unknown_df.groupby(['device_name', 'TIME'])["Size"].agg( mean="mean", std=lambda x: x.std(ddof=0), total_bytes="sum",).reset_index())
    print(f"Preprocessed {file_path}: {len(features_df)} rows.")

    return features_df

In [6]:
features_extracted_dfs = []
for file in files:
    processed_df = preprocess_csv(file, all_devices)
    if processed_df is not None and not processed_df.empty:
        features_extracted_dfs.append(processed_df)

Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-23.csv: 154260 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-24.csv: 146743 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-25.csv: 136969 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-26.csv: 140156 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-27.csv: 140052 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-28.csv: 179669 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-29.csv

In [7]:
features_df = pd.concat(features_extracted_dfs, ignore_index=True)

In [9]:
features_df["device_name"].value_counts()

device_name
Dropcam                             1477382
Withings Smart Baby Monitor          241359
Amazon Echo                          240217
Smart Things                         197749
Samsung SmartCam                     164839
Insteon Camera                       123137
Withings Aura smart sleep sensor     114800
Netatmo Welcome                      105009
Laptop                                61903
Netatmo weather station               59750
Triby Speaker                         52012
Samsung Galaxy Tab                    51915
Light Bulbs LiFX Smart Bulb           44227
Belkin wemo motion sensor             32350
TP-Link Day Night Cloud camera        26786
HP Printer                            25438
Belkin Wemo switch                    25244
MacBook                               24872
iHome                                 16909
TP-Link Smart plug                     9636
PIX-STAR Photo-frame                   9347
Android Phone                          5144
Withings Smart scale